In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy import io
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import cv2

In [ ]:
mat_file = io.loadmat('DP_Norm_Trimmed_Flat.mat')
DP_Norm_Trimmed_Flat = mat_file['DP_Norm_Trimmed_Flat']
DP_Norm_Trimmed_Flat_T=np.transpose(DP_Norm_Trimmed_Flat)

DP_Norm_Trimmed = DP_Norm_Trimmed_Flat_T.reshape(-1, 247, 247)
DP_Norm_Trimmed_Transposed = np.transpose(DP_Norm_Trimmed, axes=(0, 2, 1))
images_DP = DP_Norm_Trimmed_Transposed.reshape(-1,247,247,1)


In [ ]:
# The group indices you want to extract (1-based indexing)
group_indices = [1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 14, 16]

# Each group has 64 images
group_size = 64

# Convert group indices to zero-based (since Python uses zero-based indexing)
group_indices = [i - 1 for i in group_indices]

# Initialize an empty list to store the selected groups
selected_groups = []

# Loop through the specified group indices and extract the corresponding slices
for group in group_indices:
    start_index = group * group_size
    end_index = start_index + group_size
    selected_groups.append(images_DP[start_index:end_index, :, :, :])

# Concatenate all selected groups into one array
DP = np.concatenate(selected_groups, axis=0)


In [ ]:
def cosine_time_encoding(h, w, time_value, frequency=1.0, scale=1.0):
    x = np.linspace(0, 1, w)
    y = np.linspace(0, 1, h)
    xx, yy = np.meshgrid(x, y)
    encoding = np.cos(2 * np.pi * frequency * time_value * (xx + yy) * scale)
    return encoding

h, w = 247, 247  # Image dimensions
frequency = 3.0  # Frequency of the cosine wave
scale = 1.0  # Scaling factor

labels = [20, 19, 18, 17, 16.5, 16, 15.75, 15.5, 15, 14.5, 14, 12]


In [ ]:
total_x_data = []
total_y_data = []

for i in range(64):
    tmp=DP[i:i+64*12:64,:,:,0]
    for j in range(12):
        time_value=(labels[j]-12)/8
        cosine_encoding = cosine_time_encoding(h, w, time_value, frequency, scale)
        total_x_data.append(np.concatenate([tmp[:j,:,:], tmp[j+1:,:,:], np.reshape(cosine_encoding,(1,247,247))]))
        total_y_data.append(tmp[j,:,:])

total_x_data=np.array(total_x_data)
total_y_data=np.array(total_y_data)
total_y_data=np.reshape(total_y_data,(768,1,247,247))

np.save('total_x_near_dissociation.npy',total_x_data)
np.save('total_y_near_dissociation.npy',total_y_data)